In [1]:
%run "/Users/manojrammopati/Public/Projects/bupa_insurance_project/_01_Bronze/Jupyter Notebooks/00_bronze_data_connector.ipynb"

25/12/09 02:14:57 WARN Utils: Your hostname, Manojrams-MacBook-Air.local resolves to a loopback address: 127.0.0.1; using 192.168.0.219 instead (on interface en0)
25/12/09 02:14:57 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Ivy Default Cache set to: /Users/manojrammopati/.ivy2/cache
The jars for the packages stored in: /Users/manojrammopati/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.apache.hadoop#hadoop-azure added as a dependency
com.azure#azure-storage-blob added as a dependency
com.azure#azure-identity added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-a71a2964-2521-4780-9d2c-3cde5aa41b5c;1.0
	confs: [default]


:: loading settings :: url = jar:file:/opt/anaconda3/envs/spark_local/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


	found io.delta#delta-spark_2.12;3.1.0 in central
	found io.delta#delta-storage;3.1.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.apache.hadoop#hadoop-azure;3.3.4 in central
	found org.apache.httpcomponents#httpclient;4.5.13 in central
	found org.apache.httpcomponents#httpcore;4.4.13 in central
	found commons-logging#commons-logging;1.1.3 in central
	found commons-codec#commons-codec;1.15 in central
	found com.microsoft.azure#azure-storage;7.0.1 in central
	found com.fasterxml.jackson.core#jackson-core;2.12.7 in central
	found org.slf4j#slf4j-api;1.7.36 in central
	found com.microsoft.azure#azure-keyvault-core;1.0.0 in central
	found com.google.guava#guava;27.0-jre in central
	found com.google.guava#failureaccess;1.0 in central
	found com.google.guava#listenablefuture;9999.0-empty-to-avoid-conflict-with-guava in central
	found com.google.code.findbugs#jsr305;3.0.2 in central
	found org.checkerframework#checker-qual;2.5.2 in central
	found com.google.errorpron

Preconnectors ran successfully and connected to 'clientdatastorage' storage accounts
 'policy_df, claim_df, member_df, provider_df' are ready to use from container 'raw data'(meridian architecture of bronze) 


In [2]:
import sys
import importlib

# Add the Silver layer directory to sys.path BEFORE importing
silver_dir = "/Users/manojrammopati/Public/Projects/bupa_insurance_project/_02_Silver"
if silver_dir not in sys.path:
    sys.path.insert(0, silver_dir)

# Now import (and reload to pick up edits)
import utils_silver
from utils_silver import *

importlib.reload(utils_silver)

paths_bronze, paths_silver, paths_gold = utils_silver.build_paths(storage_account1)
print(sorted(paths_silver.keys()))
print("Imported utils_silver from", utils_silver.__file__)

✅ utils_silver.py loaded
✅ utils_silver.py loaded
['_metrics', '_quarantine', '_ref_dim_channel', '_ref_dim_product_line', '_reference', 'claims', 'members', 'policies', 'providers']
Imported utils_silver from /Users/manojrammopati/Public/Projects/bupa_insurance_project/_02_Silver/utils_silver.py


# ⭐ Cell 1 — Imports, paths, and Silver read

In [4]:
# Cell 1 — Setup & Read Silver Policies

from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql import SparkSession

from utils_silver import build_paths, dq_expect, write_metric, data_quality_summary

spark = SparkSession.builder.getOrCreate()

# Storage account (same as before)
storage_account = "clientdatastorage"

# Bronze & Silver paths from your shared utils
paths_bronze, paths_silver, paths_gold = build_paths(storage_account)

SILVER_POLICIES_PATH = paths_silver["policies"]

print("SILVER_POLICIES_PATH =", SILVER_POLICIES_PATH)

# Read Silver policies as the source for fact_policies
pol_silver = spark.read.format("delta").load(SILVER_POLICIES_PATH)

print("Silver policies rowcount:", pol_silver.count())
pol_silver.printSchema()

pol_silver.show(5, truncate=False)


SILVER_POLICIES_PATH = abfss://silverdata@clientdatastorage.dfs.core.windows.net/policies


25/12/09 02:15:34 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Silver policies rowcount: 381109
root
 |-- Policy_ID: string (nullable = true)
 |-- Customer_ID: string (nullable = true)
 |-- Product_Line: string (nullable = true)
 |-- Sum_Insured_GBP: double (nullable = true)
 |-- Annual_Premium_GBP: double (nullable = true)
 |-- Policy_Start_Date: date (nullable = true)
 |-- Policy_End_Date: date (nullable = true)
 |-- Renewal_Offered_Flag: integer (nullable = true)
 |-- Renewal_Accepted_Flag: integer (nullable = true)
 |-- Discount_Offered_%: double (nullable = true)
 |-- Channel: string (nullable = true)
 |-- dq_money_valid: integer (nullable = true)
 |-- dq_discount_valid: integer (nullable = true)
 |-- dq_renewal_valid: integer (nullable = true)
 |-- Policy_Duration_Days: integer (nullable = true)
 |-- Renewal_Conversion: integer (nullable = true)



+----------+------------+------------+------------------+------------------+-----------------+---------------+--------------------+---------------------+------------------+-------+--------------+-----------------+----------------+--------------------+------------------+
|Policy_ID |Customer_ID |Product_Line|Sum_Insured_GBP   |Annual_Premium_GBP|Policy_Start_Date|Policy_End_Date|Renewal_Offered_Flag|Renewal_Accepted_Flag|Discount_Offered_%|Channel|dq_money_valid|dq_discount_valid|dq_renewal_valid|Policy_Duration_Days|Renewal_Conversion|
+----------+------------+------------+------------------+------------------+-----------------+---------------+--------------------+---------------------+------------------+-------+--------------+-----------------+----------------+--------------------+------------------+
|P_00000003|CUST_0000003|Health      |2356477.1034685415|38294.0           |2022-01-18       |2023-01-07     |1                   |1                    |11.551851153178037|Partner|1      

# ⭐ Cell 2 — Define fact_policies schema & base selection

In [5]:
# Cell 2 — Define fact_policies base schema & select relevant columns

fact_base = pol_silver.select(
    F.col("Policy_ID").alias("Policy_ID"),
    F.col("Customer_ID").alias("Customer_ID"),
    F.col("Product_Line").alias("Product_Line"),
    F.col("Channel").alias("Channel"),
    F.col("Sum_Insured_GBP").alias("Sum_Insured_GBP"),
    F.col("Annual_Premium_GBP").alias("Annual_Premium_GBP"),
    F.col("Policy_Start_Date").alias("Policy_Start_Date"),
    F.col("Policy_End_Date").alias("Policy_End_Date"),
    F.col("Policy_Duration_Days").alias("Policy_Duration_Days"),
    F.col("Renewal_Offered_Flag").alias("Renewal_Offered_Flag"),
    F.col("Renewal_Accepted_Flag").alias("Renewal_Accepted_Flag"),
    F.col("Renewal_Conversion").alias("Renewal_Conversion"),
    # 👇 bring the discount into Gold with a nicer name
    F.col("`Discount_Offered_%`").alias("Discount_Offered_Pct"),
    # existing DQ flags
    F.col("dq_money_valid").alias("dq_money_valid"),
    F.col("dq_discount_valid").alias("dq_discount_valid"),
    F.col("dq_renewal_valid").alias("dq_renewal_valid")
)

print("fact_base rowcount:", fact_base.count())
fact_base.printSchema()
fact_base.show(5, truncate=False)



fact_base rowcount: 381109
root
 |-- Policy_ID: string (nullable = true)
 |-- Customer_ID: string (nullable = true)
 |-- Product_Line: string (nullable = true)
 |-- Channel: string (nullable = true)
 |-- Sum_Insured_GBP: double (nullable = true)
 |-- Annual_Premium_GBP: double (nullable = true)
 |-- Policy_Start_Date: date (nullable = true)
 |-- Policy_End_Date: date (nullable = true)
 |-- Policy_Duration_Days: integer (nullable = true)
 |-- Renewal_Offered_Flag: integer (nullable = true)
 |-- Renewal_Accepted_Flag: integer (nullable = true)
 |-- Renewal_Conversion: integer (nullable = true)
 |-- Discount_Offered_Pct: double (nullable = true)
 |-- dq_money_valid: integer (nullable = true)
 |-- dq_discount_valid: integer (nullable = true)
 |-- dq_renewal_valid: integer (nullable = true)

+----------+------------+------------+-------+-----------------+------------------+-----------------+---------------+--------------------+--------------------+---------------------+------------------+--

# ⭐ Cell 3 — Derived business features (segments & KPIs)

In [6]:
# Cell 3 — Derived business features for fact_policies

fact_enriched = (
    fact_base
    # Tenure bucket based on Policy_Duration_Days
    .withColumn(
        "Tenure_Band",
        F.when(F.col("Policy_Duration_Days").isNull(), F.lit("Unknown"))
         .when(F.col("Policy_Duration_Days") < 180, "0–6 months")
         .when(F.col("Policy_Duration_Days") < 365, "6–12 months")
         .when(F.col("Policy_Duration_Days") < 730, "1–2 years")
         .otherwise("2+ years")
    )
    # Premium band
    .withColumn(
        "Premium_Band",
        F.when(F.col("Annual_Premium_GBP") < 250, "Low (<250)")
         .when(F.col("Annual_Premium_GBP") < 750, "Medium (250–750)")
         .when(F.col("Annual_Premium_GBP") < 1500, "High (750–1500)")
         .otherwise("Very High (1500+)")
    )
    # Discount band — now using Discount_Offered_Pct
    .withColumn(
        "Discount_Band",
        F.when(F.col("Discount_Offered_Pct").isNull(), "No discount info")
         .when(F.col("Discount_Offered_Pct") == 0, "No discount")
         .when(F.col("Discount_Offered_Pct") <= 10, "0–10%")
         .when(F.col("Discount_Offered_Pct") <= 20, "10–20%")
         .otherwise("20%+")
    )
    # Simple renewal outcome label
    .withColumn(
        "Renewal_Outcome",
        F.when((F.col("Renewal_Offered_Flag") == 1) & (F.col("Renewal_Accepted_Flag") == 1), "Renewed")
         .when((F.col("Renewal_Offered_Flag") == 1) & (F.col("Renewal_Accepted_Flag") == 0), "Not renewed")
         .when(F.col("Renewal_Offered_Flag").isNull(), "No renewal info")
         .otherwise("No offer")
    )
)

fact_enriched.show(5, truncate=False)
fact_enriched.printSchema()



+----------+------------+------------+-------+-----------------+------------------+-----------------+---------------+--------------------+--------------------+---------------------+------------------+--------------------+--------------+-----------------+----------------+-----------+-----------------+-------------+---------------+
|Policy_ID |Customer_ID |Product_Line|Channel|Sum_Insured_GBP  |Annual_Premium_GBP|Policy_Start_Date|Policy_End_Date|Policy_Duration_Days|Renewal_Offered_Flag|Renewal_Accepted_Flag|Renewal_Conversion|Discount_Offered_Pct|dq_money_valid|dq_discount_valid|dq_renewal_valid|Tenure_Band|Premium_Band     |Discount_Band|Renewal_Outcome|
+----------+------------+------------+-------+-----------------+------------------+-----------------+---------------+--------------------+--------------------+---------------------+------------------+--------------------+--------------+-----------------+----------------+-----------+-----------------+-------------+---------------+
|P_0

# ⭐ Cell 4 — Data quality checks on fact_policies

In [7]:
# Cell 4 — DQ checks on fact_policies

# Re-use dq_expect from utils_silver
# Signature: dq_expect(df, name, expr, severity, table_label, paths_silver)

dq_expect(
    fact_enriched,
    "pk_not_null",
    "Policy_ID IS NOT NULL",
    "error",
    "fact_policies",
    paths_silver
)

dq_expect(
    fact_enriched,
    "money_non_negative",
    "coalesce(Annual_Premium_GBP,0) >= 0 AND coalesce(Sum_Insured_GBP,0) >= 0",
    "error",
    "fact_policies",
    paths_silver
)

dq_expect(
    fact_enriched,
    "discount_bounds",
    "`Discount_Offered_Pct` IS NULL OR (`Discount_Offered_Pct` BETWEEN 0 AND 100)",
    "error",
    "fact_policies",
    paths_silver
)

dq_expect(
    fact_enriched,
    "date_order",
    "Policy_Start_Date IS NULL OR Policy_End_Date IS NULL OR Policy_End_Date >= Policy_Start_Date",
    "error",
    "fact_policies",
    paths_silver
)

print("DQ checks completed for fact_policies.")


✅ DQ PASS [fact_policies] pk_not_null


✅ DQ PASS [fact_policies] money_non_negative
✅ DQ PASS [fact_policies] discount_bounds
✅ DQ PASS [fact_policies] date_order
DQ checks completed for fact_policies.


# ⭐ Cell 5 — Final fact_policies projection (what goes to Gold)

In [8]:
# Cell 5 — Final projection for Gold fact_policies

fact_policies = fact_enriched.select(
    "Policy_ID",
    "Customer_ID",
    "Product_Line",
    "Channel",
    "Sum_Insured_GBP",
    "Annual_Premium_GBP",
    "Policy_Start_Date",
    "Policy_End_Date",
    "Policy_Duration_Days",
    "Renewal_Offered_Flag",
    "Renewal_Accepted_Flag",
    "Renewal_Conversion",
    "Tenure_Band",
    "Premium_Band",
    "Discount_Band",
    "Renewal_Outcome",
    "dq_money_valid",
    "dq_discount_valid",
    "dq_renewal_valid"
)

print("fact_policies rows:", fact_policies.count())
fact_policies.show(10, truncate=False)


fact_policies rows: 381109
+----------+------------+------------+-------+------------------+------------------+-----------------+---------------+--------------------+--------------------+---------------------+------------------+-----------+-----------------+-------------+---------------+--------------+-----------------+----------------+
|Policy_ID |Customer_ID |Product_Line|Channel|Sum_Insured_GBP   |Annual_Premium_GBP|Policy_Start_Date|Policy_End_Date|Policy_Duration_Days|Renewal_Offered_Flag|Renewal_Accepted_Flag|Renewal_Conversion|Tenure_Band|Premium_Band     |Discount_Band|Renewal_Outcome|dq_money_valid|dq_discount_valid|dq_renewal_valid|
+----------+------------+------------+-------+------------------+------------------+-----------------+---------------+--------------------+--------------------+---------------------+------------------+-----------+-----------------+-------------+---------------+--------------+-----------------+----------------+
|P_00000004|CUST_0000004|Health      

# ⭐ Cell 6 — Optional: attach audit columns

In [9]:
# Cell 6 — Optional audit columns

from utils_silver import add_audit_columns

fact_policies_audit = add_audit_columns(fact_policies, source_name="silver_policies_to_gold_fact")

fact_policies_audit.printSchema()
fact_policies_audit.show(5, truncate=False)


root
 |-- Policy_ID: string (nullable = true)
 |-- Customer_ID: string (nullable = true)
 |-- Product_Line: string (nullable = true)
 |-- Channel: string (nullable = true)
 |-- Sum_Insured_GBP: double (nullable = true)
 |-- Annual_Premium_GBP: double (nullable = true)
 |-- Policy_Start_Date: date (nullable = true)
 |-- Policy_End_Date: date (nullable = true)
 |-- Policy_Duration_Days: integer (nullable = true)
 |-- Renewal_Offered_Flag: integer (nullable = true)
 |-- Renewal_Accepted_Flag: integer (nullable = true)
 |-- Renewal_Conversion: integer (nullable = true)
 |-- Tenure_Band: string (nullable = false)
 |-- Premium_Band: string (nullable = false)
 |-- Discount_Band: string (nullable = false)
 |-- Renewal_Outcome: string (nullable = false)
 |-- dq_money_valid: integer (nullable = true)
 |-- dq_discount_valid: integer (nullable = true)
 |-- dq_renewal_valid: integer (nullable = true)
 |-- created_at: timestamp (nullable = false)
 |-- source_system: string (nullable = false)
 |-- re

# ⭐ Cell 7 — Write to Gold Delta path

In [10]:
# Cell 7 — Write Gold fact_policies to Delta (golddata container)

GOLD_CONTAINER = "golddata"
GOLD_POLICIES_PATH = f"abfss://{GOLD_CONTAINER}@{storage_account}.dfs.core.windows.net/fact_policies"

print("GOLD_POLICIES_PATH =", GOLD_POLICIES_PATH)

(
    fact_policies_audit
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(GOLD_POLICIES_PATH)
)

print("✅ fact_policies written to:", GOLD_POLICIES_PATH)


GOLD_POLICIES_PATH = abfss://golddata@clientdatastorage.dfs.core.windows.net/fact_policies


25/12/05 12:26:32 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


✅ fact_policies written to: abfss://golddata@clientdatastorage.dfs.core.windows.net/fact_policies


# ⭐ Cell 8 — Register in bupa_gold schema

In [11]:
# Cell 8 — Register table in bupa_gold.fact_policies

spark.sql("CREATE DATABASE IF NOT EXISTS bupa_gold")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS bupa_gold.fact_policies
USING DELTA
LOCATION '{GOLD_POLICIES_PATH}'
""")

print("✅ Registered table: bupa_gold.fact_policies")

spark.table("bupa_gold.fact_policies").show(10, truncate=False)


✅ Registered table: bupa_gold.fact_policies


+----------+------------+------------+-------+------------------+------------------+-----------------+---------------+--------------------+--------------------+---------------------+------------------+-----------+-----------------+-------------+---------------+--------------+-----------------+----------------+--------------------------+----------------------------+----------------------------------------------------------------+
|Policy_ID |Customer_ID |Product_Line|Channel|Sum_Insured_GBP   |Annual_Premium_GBP|Policy_Start_Date|Policy_End_Date|Policy_Duration_Days|Renewal_Offered_Flag|Renewal_Accepted_Flag|Renewal_Conversion|Tenure_Band|Premium_Band     |Discount_Band|Renewal_Outcome|dq_money_valid|dq_discount_valid|dq_renewal_valid|created_at                |source_system               |record_hash                                                     |
+----------+------------+------------+-------+------------------+------------------+-----------------+---------------+----------------

# sample data from "bupa_gold.fact_policies"

In [23]:
# Read Gold fact_policies table
fact_policies = spark.table("bupa_gold.fact_policies")

# Display schema and sample data
fact_policies.printSchema()
fact_policies.show(10, truncate=False)

print("Row count:", fact_policies.count())

root
 |-- Policy_ID: string (nullable = true)
 |-- Customer_ID: string (nullable = true)
 |-- Product_Line: string (nullable = true)
 |-- Channel: string (nullable = true)
 |-- Sum_Insured_GBP: double (nullable = true)
 |-- Annual_Premium_GBP: double (nullable = true)
 |-- Policy_Start_Date: date (nullable = true)
 |-- Policy_End_Date: date (nullable = true)
 |-- Policy_Duration_Days: integer (nullable = true)
 |-- Renewal_Offered_Flag: integer (nullable = true)
 |-- Renewal_Accepted_Flag: integer (nullable = true)
 |-- Renewal_Conversion: integer (nullable = true)
 |-- Tenure_Band: string (nullable = true)
 |-- Premium_Band: string (nullable = true)
 |-- Discount_Band: string (nullable = true)
 |-- Renewal_Outcome: string (nullable = true)
 |-- dq_money_valid: integer (nullable = true)
 |-- dq_discount_valid: integer (nullable = true)
 |-- dq_renewal_valid: integer (nullable = true)
 |-- created_at: timestamp (nullable = true)
 |-- source_system: string (nullable = true)
 |-- record_h

+----------+------------+------------+-------+------------------+------------------+-----------------+---------------+--------------------+--------------------+---------------------+------------------+-----------+-----------------+-------------+---------------+--------------+-----------------+----------------+--------------------------+----------------------------+----------------------------------------------------------------+
|Policy_ID |Customer_ID |Product_Line|Channel|Sum_Insured_GBP   |Annual_Premium_GBP|Policy_Start_Date|Policy_End_Date|Policy_Duration_Days|Renewal_Offered_Flag|Renewal_Accepted_Flag|Renewal_Conversion|Tenure_Band|Premium_Band     |Discount_Band|Renewal_Outcome|dq_money_valid|dq_discount_valid|dq_renewal_valid|created_at                |source_system               |record_hash                                                     |
+----------+------------+------------+-------+------------------+------------------+-----------------+---------------+----------------

# ⭐ Cell 9 — Metrics (rowcounts, distinct policies)

In [12]:
# Cell 9 — Metrics logging for fact_policies

from utils_silver import write_metric

write_metric(
    spark=spark,
    name="rowcount_fact_policies",
    value=fact_policies_audit.count(),
    context="fact_policies_gold",
    paths_silver=paths_silver
)

write_metric(
    spark=spark,
    name="distinct_policy_ids_fact_policies",
    value=fact_policies_audit.select("Policy_ID").distinct().count(),
    context="fact_policies_gold",
    paths_silver=paths_silver
)

metrics_df = spark.read.format("delta").load(paths_silver["_metrics"])
metrics_df.orderBy(F.col("ts").desc()).show(20, truncate=False)


[METRIC] rowcount_fact_policies=381109 ctx=fact_policies_gold


[METRIC] distinct_policy_ids_fact_policies=381109 ctx=fact_policies_gold


+---------------------------------+------+------------------+--------------------------+
|metric                           |value |context           |ts                        |
+---------------------------------+------+------------------+--------------------------+
|distinct_policy_ids_fact_policies|381109|fact_policies_gold|2025-12-05 12:26:53.739201|
|rowcount_fact_policies           |381109|fact_policies_gold|2025-12-05 12:26:45.024747|
|distinct_claim_ids_fact_claims   |558211|gold_fact_claims  |2025-12-05 12:23:28.437382|
|rowcount_fact_claims             |558211|gold_fact_claims  |2025-12-05 12:23:18.940506|
|distinct_member_ids_fact_members |381109|fact_members_gold |2025-12-04 22:09:52.334536|
|rowcount_fact_members            |381109|fact_members_gold |2025-12-04 22:09:37.352454|
|distinct_policy_ids_fact_policies|381109|fact_policies_gold|2025-12-04 20:56:22.733116|
|rowcount_fact_policies           |381109|fact_policies_gold|2025-12-04 20:56:13.37095 |
|distinct_claim_ids_f

# ⭐ Cell 10 — Quick business sanity checks

In [13]:
# Cell 10 — Quick business sanity checks

fact_p = spark.table("bupa_gold.fact_policies")

print("Total policies in Gold:", fact_p.count())
print("Distinct Policy_ID:", fact_p.select("Policy_ID").distinct().count())

print("\nTenure_Band distribution:")
(
    fact_p.groupBy("Tenure_Band")
          .agg(F.count("*").alias("n_policies"))
          .orderBy(F.desc("n_policies"))
          .show(truncate=False)
)

print("\nRenewal_Outcome distribution:")
(
    fact_p.groupBy("Renewal_Outcome")
          .agg(F.count("*").alias("n_policies"))
          .orderBy(F.desc("n_policies"))
          .show(truncate=False)
)

print("\nProduct_Line x Channel matrix:")
(
    fact_p.groupBy("Product_Line", "Channel")
          .agg(F.count("*").alias("n_policies"))
          .orderBy("Product_Line", "Channel")
          .show(50, truncate=False)
)


Total policies in Gold: 381109


Distinct Policy_ID: 381109

Tenure_Band distribution:


+-----------+----------+
|Tenure_Band|n_policies|
+-----------+----------+
|6–12 months|195780    |
|1–2 years  |185329    |
+-----------+----------+


Renewal_Outcome distribution:


+---------------+----------+
|Renewal_Outcome|n_policies|
+---------------+----------+
|Renewed        |199315    |
|No offer       |95670     |
|Not renewed    |86124     |
+---------------+----------+


Product_Line x Channel matrix:


+------------+-------+----------+
|Product_Line|Channel|n_policies|
+------------+-------+----------+
|Accident    |Agent  |14        |
|Accident    |Broker |6         |
|Accident    |Online |6         |
|Accident    |Partner|6427      |
|Dental      |Agent  |137       |
|Dental      |Broker |157       |
|Dental      |Online |323       |
|Dental      |Partner|106367    |
|Health      |Agent  |311       |
|Health      |Broker |295       |
|Health      |Online |635       |
|Health      |Partner|227352    |
|Motor       |Agent  |65        |
|Motor       |Broker |53        |
|Motor       |Online |108       |
|Motor       |Partner|38453     |
|Travel      |Broker |1         |
|Travel      |Online |2         |
|Travel      |Partner|397       |
+------------+-------+----------+



# 🔍 Analysis Cells A–D (what’s happening to the data)

# 🅰 Cell A — Bronze vs Silver vs Gold row counts

In [14]:
# Cell A — Bronze vs Silver vs Gold row counts

bronze_policies = spark.read.format("delta").load(paths_bronze["policies"])
silver_policies = pol_silver
gold_policies   = fact_policies_audit

summary_counts = {
    "bronze_rows": bronze_policies.count(),
    "silver_rows": silver_policies.count(),
    "gold_rows":   gold_policies.count(),
    "bronze_distinct_policy_id": bronze_policies.select("Policy_ID").distinct().count(),
    "silver_distinct_policy_id": silver_policies.select("Policy_ID").distinct().count(),
    "gold_distinct_policy_id":   gold_policies.select("Policy_ID").distinct().count()
}

print("Rowcount & distinct Policy_ID comparison:")
for k, v in summary_counts.items():
    print(f"  {k}: {v}")


Rowcount & distinct Policy_ID comparison:
  bronze_rows: 381109
  silver_rows: 381109
  gold_rows: 381109
  bronze_distinct_policy_id: 381109
  silver_distinct_policy_id: 381109
  gold_distinct_policy_id: 381109


# 🅱 Cell B — Check that core fields stayed stable

In [15]:
# Cell B — Check core features unchanged from Silver to Gold

core_cols = [
    "Policy_ID",
    "Customer_ID",
    "Product_Line",
    "Channel",
    "Sum_Insured_GBP",
    "Annual_Premium_GBP",
    "Policy_Start_Date",
    "Policy_End_Date"
]

sv = pol_silver.select(core_cols).dropDuplicates(["Policy_ID"]).alias("sv")
gd = fact_policies_audit.select(core_cols).dropDuplicates(["Policy_ID"]).alias("gd")

joined = sv.join(gd, on="Policy_ID", how="inner")

for col in core_cols:
    if col == "Policy_ID":
        continue
    diff_cnt = (
        joined
        .filter((F.col(f"sv.{col}") != F.col(f"gd.{col}")) |
                (F.col(f"sv.{col}").isNull() & F.col(f"gd.{col}").isNotNull()) |
                (F.col(f"sv.{col}").isNotNull() & F.col(f"gd.{col}").isNull()))
        .count()
    )
    if diff_cnt == 0:
        print(f"✅ '{col}' unchanged between Silver and Gold (by Policy_ID).")
    else:
        print(f"⚠️ '{col}' differs for {diff_cnt} policies between Silver and Gold.")


✅ 'Customer_ID' unchanged between Silver and Gold (by Policy_ID).


✅ 'Product_Line' unchanged between Silver and Gold (by Policy_ID).


✅ 'Channel' unchanged between Silver and Gold (by Policy_ID).


✅ 'Sum_Insured_GBP' unchanged between Silver and Gold (by Policy_ID).


✅ 'Annual_Premium_GBP' unchanged between Silver and Gold (by Policy_ID).


✅ 'Policy_Start_Date' unchanged between Silver and Gold (by Policy_ID).


✅ 'Policy_End_Date' unchanged between Silver and Gold (by Policy_ID).


# 🅲 Cell C — New features in fact_policies and why

In [16]:
# Cell C — List new features in Gold fact_policies and sample values

sv_cols = set(pol_silver.columns)
gd_cols = set(fact_policies_audit.columns)

new_in_gold = sorted(gd_cols - sv_cols)
dropped_in_gold = sorted(sv_cols - gd_cols)
common_cols = sorted(gd_cols & sv_cols)

print("New columns in Gold fact_policies:", new_in_gold)
print("Dropped columns in Gold fact_policies:", dropped_in_gold)
print("Common columns:", len(common_cols))

# Show samples for new columns with high-level explanation
for col in new_in_gold:
    print(f"\n▶ Sample values for new Gold column: {col}")
    fact_policies_audit.select("Policy_ID", col).show(5, truncate=False)


New columns in Gold fact_policies: ['Discount_Band', 'Premium_Band', 'Renewal_Outcome', 'Tenure_Band', 'created_at', 'record_hash', 'source_system']
Dropped columns in Gold fact_policies: ['Discount_Offered_%']
Common columns: 15

▶ Sample values for new Gold column: Discount_Band


+----------+-------------+
|Policy_ID |Discount_Band|
+----------+-------------+
|P_00000004|10–20%       |
|P_00000009|10–20%       |
|P_00000019|10–20%       |
|P_00000037|0–10%        |
|P_00000043|10–20%       |
+----------+-------------+
only showing top 5 rows


▶ Sample values for new Gold column: Premium_Band


+----------+-----------------+
|Policy_ID |Premium_Band     |
+----------+-----------------+
|P_00000004|Very High (1500+)|
|P_00000009|Very High (1500+)|
|P_00000019|Very High (1500+)|
|P_00000037|Very High (1500+)|
|P_00000043|Very High (1500+)|
+----------+-----------------+
only showing top 5 rows


▶ Sample values for new Gold column: Renewal_Outcome
+----------+---------------+
|Policy_ID |Renewal_Outcome|
+----------+---------------+
|P_00000004|Renewed        |
|P_00000009|Renewed        |
|P_00000019|Not renewed    |
|P_00000037|Renewed        |
|P_00000043|Renewed        |
+----------+---------------+
only showing top 5 rows


▶ Sample values for new Gold column: Tenure_Band


+----------+-----------+
|Policy_ID |Tenure_Band|
+----------+-----------+
|P_00000004|1–2 years  |
|P_00000009|6–12 months|
|P_00000019|6–12 months|
|P_00000037|6–12 months|
|P_00000043|6–12 months|
+----------+-----------+
only showing top 5 rows


▶ Sample values for new Gold column: created_at
+----------+--------------------------+
|Policy_ID |created_at                |
+----------+--------------------------+
|P_00000004|2025-12-05 12:27:30.282657|
|P_00000009|2025-12-05 12:27:30.282657|
|P_00000019|2025-12-05 12:27:30.282657|
|P_00000037|2025-12-05 12:27:30.282657|
|P_00000043|2025-12-05 12:27:30.282657|
+----------+--------------------------+
only showing top 5 rows


▶ Sample values for new Gold column: record_hash
+----------+----------------------------------------------------------------+
|Policy_ID |record_hash                                                     |
+----------+----------------------------------------------------------------+
|P_00000004|fee089190292e1b85e9b

# 🅳 Cell D — KPIs for business stakeholders

In [17]:
# Cell D — Business KPIs from fact_policies

fp = fact_policies_audit

print("Average premium & sum insured by Product_Line:")
(
    fp.groupBy("Product_Line")
      .agg(
          F.count("*").alias("n_policies"),
          F.avg("Annual_Premium_GBP").alias("avg_premium"),
          F.avg("Sum_Insured_GBP").alias("avg_sum_insured")
      )
      .orderBy("Product_Line")
      .show(truncate=False)
)

print("\nRenewal conversion by Product_Line:")
(
    fp.groupBy("Product_Line")
      .agg(
          F.count("*").alias("n_policies"),
          F.avg(F.col("Renewal_Conversion").cast("double")).alias("renewal_conversion_rate")
      )
      .orderBy("Product_Line")
      .show(truncate=False)
)

print("\nRenewal conversion by Channel:")
(
    fp.groupBy("Channel")
      .agg(
          F.count("*").alias("n_policies"),
          F.avg(F.col("Renewal_Conversion").cast("double")).alias("renewal_conversion_rate")
      )
      .orderBy("Channel")
      .show(truncate=False)
)


Average premium & sum insured by Product_Line:


+------------+----------+------------------+------------------+
|Product_Line|n_policies|avg_premium       |avg_sum_insured   |
+------------+----------+------------------+------------------+
|Accident    |6453      |30780.595536959554|1768752.290405056 |
|Dental      |106984    |30634.113278621102|1761571.7542735534|
|Health      |228593    |30523.16170223935 |1755214.8452213777|
|Motor       |38679     |30592.356007135655|1757211.9909545125|
|Travel      |400       |29284.875         |1687246.1612595061|
+------------+----------+------------------+------------------+


Renewal conversion by Product_Line:


+------------+----------+-----------------------+
|Product_Line|n_policies|renewal_conversion_rate|
+------------+----------+-----------------------+
|Accident    |6453      |0.6940592009935831     |
|Dental      |106984    |0.6982589703588143     |
|Health      |228593    |0.6983529439230518     |
|Motor       |38679     |0.6984539656128674     |
|Travel      |400       |0.7090301003344481     |
+------------+----------+-----------------------+


Renewal conversion by Channel:
+-------+----------+-----------------------+
|Channel|n_policies|renewal_conversion_rate|
+-------+----------+-----------------------+
|Agent  |527       |0.6947890818858561     |
|Broker |512       |0.6966580976863753     |
|Online |1074      |0.6942046855733662     |
|Partner|378996    |0.6982940853168731     |
+-------+----------+-----------------------+



In [18]:
print("1️⃣ Renewal conversion rate")
spark.table("bupa_gold.fact_policies") \
    .groupBy("Renewal_Outcome") \
    .count().show()

print("2️⃣ Premium segmentation")
spark.table("bupa_gold.fact_policies") \
    .groupBy("Premium_Band").count().orderBy("Premium_Band").show()

print("3️⃣ Product Line performance")
spark.table("bupa_gold.fact_policies") \
    .groupBy("Product_Line") \
    .agg(
        F.count("*").alias("n_policies"),
        F.avg("Annual_Premium_GBP").alias("avg_premium")
    ).show()

print("4️⃣ Channel mix (Broker vs Online vs Agent)")
spark.table("bupa_gold.fact_policies") \
    .groupBy("Channel") \
    .count().show()

1️⃣ Renewal conversion rate


+---------------+------+
|Renewal_Outcome| count|
+---------------+------+
|    Not renewed| 86124|
|        Renewed|199315|
|       No offer| 95670|
+---------------+------+

2️⃣ Premium segmentation


+-----------------+------+
|     Premium_Band| count|
+-----------------+------+
|Very High (1500+)|381109|
+-----------------+------+

3️⃣ Product Line performance


+------------+----------+------------------+
|Product_Line|n_policies|       avg_premium|
+------------+----------+------------------+
|       Motor|     38679|30592.356007135655|
|      Travel|       400|         29284.875|
|      Health|    228593| 30523.16170223935|
|      Dental|    106984|30634.113278621102|
|    Accident|      6453|30780.595536959554|
+------------+----------+------------------+

4️⃣ Channel mix (Broker vs Online vs Agent)


+-------+------+
|Channel| count|
+-------+------+
|Partner|378996|
| Broker|   512|
|  Agent|   527|
| Online|  1074|
+-------+------+

